# Data Preprocessing

This notebook constructs `master_scaled_replicated.csv` from raw NHS and ONS data sources, replicating the dataset used by James et al. (2023). Each variable is scaled per 10,000 population to ensure comparability across CCGs of different sizes.

The NHS publishes different datasets at different geographic levels (at CCG level, trust or area level). If data is not available at CCG level, it is stratified down using population share. When CCG cannot be mapped due to boundary changes, values are taken from the authors' original `master_scaled.csv`.

| Variable | Source | CCGs independent | CCGs fallback |
|---|---|---|---|
| `gp_appt_available` | NHS Digital - Appointments in General Practice | 73 | 0 |
| `amb_sys_made`, `amb_sys_answered` | NHS England - AmbSYS Time Series | 64 | 9 |
| `111_111_offered`, `111_111_answered` | NHS England - NHS 111 MDS | 48 | 25 |
| `population` | ONS Mid-Year Population Estimates | 64 | 9 |
| `People`, `Places`, `Lives` | ONS Health Index (via LSOA crosswalk - see separate notebook) | 73 | 0 |
| `ae_attendances_attendances` | Authors' master_scaled.csv | 0 | 73 |

Import Statements

In [2]:
import os, glob, re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Paths

In [3]:
BASE_PATH = '../data/'
OUTPUT_PATH = '../new_data/'

In [4]:
AMB_PATH     = os.path.join(BASE_PATH, 'ambulance')
GP_PATH      = os.path.join(BASE_PATH, 'GP')
POP_PATH     = os.path.join(BASE_PATH, 'population')
NHS111_PATH  = os.path.join(BASE_PATH, '111')
MASTER_PATH  = os.path.join(BASE_PATH, 'master_scaled.csv')
AMB_TS       = os.path.join(AMB_PATH, 'AmbSYS-Time-Series-to-20260531-k58VJ.xlsx')
NHS111_FILE  = os.path.join(NHS111_PATH, '20200409-NHS-111-MDS-time-series-to-March-2020.xlsx')
HI_CCG_FILE  = os.path.join(BASE_PATH, 'index', 'health_index_ccg.csv')

## 1. Load master_scaled.csv and Health Index

ED attendances have no public CCG-level source. They are published at provider (hospital trust) level only. These are taken directly from the authors' file. The Health Index crosswalk (People, Places, Lives) is built in a separate notebook using LSOA geographic lookup data and loaded here.

In [5]:
master = pd.read_csv(MASTER_PATH, index_col=0)
master_ccg_list = master['ccg'].unique()
print('Master shape:', master.shape, '| CCGs:', master['ccg'].nunique())

ed           = master[['ccg','month','year','ae_attendances_attendances']].copy()
health_index = pd.read_csv(HI_CCG_FILE)
gp_master    = master[['ccg','month','year','gp_appt_available']].copy()

print('Health Index shape:', health_index.shape)
print(health_index.head(3))

Master shape: (1465, 13) | CCGs: 73
Health Index shape: (146, 5)
   ccg    year  People  Lives  Places
0  00Q  2018.0    97.2   94.4    99.7
1  00Q  2019.0    96.0   94.6    99.5
2  00R  2018.0    85.8   88.0    97.6


## 2. Population

Population estimates come from ONS Mid-Year Population Estimates at CCG level (files SAPE21 for 2018 and SAPE22 for 2019). These use ONS CCG codes (E38...) which need mapping to NHS codes (e.g. 00Q). The mapping comes from the CCG lookup sheet inside the AmbSYS time series file, which has both code types side by side.

9 CCGs use non-standard NHS codes that do not appear in any public lookup, these fall back to the authors' values. A small number of CCGs only appear in one year due to boundary changes; their population is forward/backward filled to cover both years.

In [6]:
def extract_population(filepath, year):
    df = pd.read_excel(filepath, sheet_name=f'Mid-{year} Persons', header=None)
    ccg_rows = df[df[0].astype(str).str.startswith('E38')].copy()
    pop = pd.DataFrame()
    pop['ccg_ons'] = ccg_rows[0].values
    pop['population_raw'] = pd.to_numeric(ccg_rows[4], errors='coerce').values
    pop['year'] = year
    pop['population_10k'] = pop['population_raw'] / 10000
    return pop

In [7]:
pop_2018_raw = extract_population(
    os.path.join(POP_PATH,'SAPE21DT5-mid-2018_on_2019-ccg-syoa-estimates-formatted_corrected.xlsx'), 2018)
pop_2019_raw = extract_population(
    os.path.join(POP_PATH,'SAPE22DT6a-mid-2019-ccg-2020-estimates-unformatted.xlsx'), 2019)
pop_raw = pd.concat([pop_2018_raw, pop_2019_raw], ignore_index=True)

In [8]:
# map ONS codes to NHS codes using the ambulance CCG lookup
ccg_lookup = pd.read_excel(AMB_TS, sheet_name='CCG lookup', header=None)
ccg_lookup = ccg_lookup.iloc[2:,[1,3]].copy()
ccg_lookup.columns = ['ccg_nhs','ccg_ons']
ccg_lookup = ccg_lookup.dropna(subset=['ccg_nhs'])
ccg_lookup = ccg_lookup[ccg_lookup['ccg_nhs'].str.match(r'^[0-9A-Z]{3}$', na=False)]

In [9]:
pop_raw = pop_raw.merge(ccg_lookup, on='ccg_ons', how='left')
pop_independent = pop_raw[pop_raw['ccg_nhs'].isin(master_ccg_list)].copy()
pop_independent = pop_independent[['ccg_nhs','year','population_10k']].copy()
pop_independent.columns = ['ccg','year','population']

independent_pop_ccgs = set(pop_independent['ccg'].unique())
fallback_pop_ccgs = set(master_ccg_list) - independent_pop_ccgs
print(f'Population — independent: {len(independent_pop_ccgs)}, fallback: {len(fallback_pop_ccgs)}')

Population — independent: 64, fallback: 9


In [10]:
pop_fallback = master[master['ccg'].isin(fallback_pop_ccgs)][['ccg','year','population']].drop_duplicates()
pop_combined = pd.concat([pop_independent, pop_fallback], ignore_index=True)

# create a complete CCG x year grid and fill any gaps
full_grid = pd.MultiIndex.from_product([pop_combined['ccg'].unique(),[2018,2019]], names=['ccg','year'])
pop_complete = pop_combined.set_index(['ccg','year']).reindex(full_grid).reset_index()
pop_complete['population'] = pop_complete.groupby('ccg')['population'].transform(lambda x: x.ffill().bfill())

# some CCGs had NaN population even after the grid fill — top up from master
master_pop_all = master[['ccg','year','population']].drop_duplicates()
pop_complete = pop_complete.merge(master_pop_all, on=['ccg','year'], how='left', suffixes=('','_master'))
pop_complete['population'] = pop_complete['population'].fillna(pop_complete['population_master'])
pop_complete = pop_complete.drop(columns=['population_master'])
pop_complete['population'] = pop_complete.groupby('ccg')['population'].transform(lambda x: x.ffill().bfill())

print(f'Population final shape: {pop_complete.shape} | Missing: {pop_complete["population"].isna().sum()}')

Population final shape: (146, 3) | Missing: 0


## 3. Ambulance

Ambulance data comes from NHS England's AmbSYS time series, which reports monthly figures for each of the 11 ambulance trusts in England. As trusts cover multiple CCGs, the trust-level figures need splitting down to CCG level. This is done by population share. 

If a CCG has 30% of a trust's total population, it is given 30% of that trust's calls. The CCG-to-trust mapping is included in the AmbSYS file itself. After correcting for naming inconsistencies in the trust names (SECAmb vs SECAMB, SWAS vs SWASFT), 64 of 73 CCGs can be mapped. The remaining 9 use non-standard codes and fall back to the authors' values.

In [11]:
def extract_ambulance_timeseries(filepath):
    df = pd.read_excel(filepath, sheet_name='Raw', header=None)
    data = df.iloc[5:,  [1,2,5,7,8]].copy()
    data.columns = ['year_str','month_raw','trust_code','amb_sys_made','amb_sys_answered']
    trust_codes = ['RX9','RYC','R1F','RRU','RX6','RX7','RYE','RYD','RYF','RYA','RX8']
    data = data[data['trust_code'].isin(trust_codes)].copy()
    jan_mar = ['JANUARY','FEBRUARY','MARCH']
    data['month_raw'] = data['month_raw'].str.upper()
    def parse_year(row):
        f = int(str(row['year_str'])[:4])
        return f+1 if row['month_raw'] in jan_mar else f
    data['year'] = data.apply(parse_year, axis=1)
    month_map = {'JANUARY':'Jan','FEBRUARY':'Feb','MARCH':'Mar','APRIL':'Apr',
                 'MAY':'May','JUNE':'Jun','JULY':'Jul','AUGUST':'Aug',
                 'SEPTEMBER':'Sep','OCTOBER':'Oct','NOVEMBER':'Nov','DECEMBER':'Dec'}
    data['month'] = data['month_raw'].map(month_map)
    data = data[data['year'].isin([2018,2019])]
    data['amb_sys_made']     = pd.to_numeric(data['amb_sys_made'], errors='coerce')
    data['amb_sys_answered'] = pd.to_numeric(data['amb_sys_answered'], errors='coerce')
    return data[['trust_code','month','year','amb_sys_made','amb_sys_answered']].reset_index(drop=True)

In [12]:
amb_trust = extract_ambulance_timeseries(AMB_TS)
print(f'Trust-level ambulance shape: {amb_trust.shape}')

trust_name_to_code = {'EEAST':'RYC','EMAS':'RX9','LAS':'RRU','NEAS':'RX6','NWAS':'RX7','SCAS':'RYE',
                      'SECAmb':'RYD','SWAS':'RYF','WMAS':'RYA','YAS':'RX8','IoW':'R1F'}

Trust-level ambulance shape: (264, 5)


In [13]:
ccg_trust_map = pd.read_excel(AMB_TS, sheet_name='CCG lookup', header=None)
ccg_trust_map = ccg_trust_map.iloc[2:,[1,4]].copy()
ccg_trust_map.columns = ['ccg','trust_name']
ccg_trust_map = ccg_trust_map.dropna(subset=['ccg'])
ccg_trust_map = ccg_trust_map[ccg_trust_map['ccg'].str.match(r'^[0-9A-Z]{3}$', na=False)]
ccg_trust_map['trust_code'] = ccg_trust_map['trust_name'].map(trust_name_to_code)
ccg_trust_map = ccg_trust_map[ccg_trust_map['ccg'].isin(master_ccg_list)]
ccg_trust_map = ccg_trust_map[ccg_trust_map['trust_code'].notna()]
print(f'CCGs with trust mapping: {ccg_trust_map["ccg"].nunique()}')

CCGs with trust mapping: 64


In [14]:
# merge CCG-trust mapping with trust-level data, then stratify by population share
amb_ccg = ccg_trust_map[['ccg','trust_code']].merge(amb_trust, on='trust_code', how='left')
amb_ccg = amb_ccg.merge(pop_complete, on=['ccg','year'], how='left')
trust_pop = amb_ccg.groupby(['trust_code','month','year'])['population'].transform('sum')
amb_ccg['pop_share'] = amb_ccg['population'] / trust_pop
amb_ccg['amb_sys_made']     = amb_ccg['amb_sys_made']     * amb_ccg['pop_share'] / amb_ccg['population']
amb_ccg['amb_sys_answered'] = amb_ccg['amb_sys_answered'] * amb_ccg['pop_share'] / amb_ccg['population']

In [15]:
amb_independent = amb_ccg[['ccg','month','year','amb_sys_made','amb_sys_answered']].dropna()
independent_amb_ccgs = set(amb_independent['ccg'].unique())
fallback_amb_ccgs = set(master_ccg_list) - independent_amb_ccgs
print(f'Ambulance — independent: {len(independent_amb_ccgs)}, fallback: {len(fallback_amb_ccgs)}')

amb_fallback = master[master['ccg'].isin(fallback_amb_ccgs)][['ccg','month','year','amb_sys_made','amb_sys_answered']].copy()
amb_final = pd.concat([amb_independent, amb_fallback], ignore_index=True)
print(f'Ambulance final shape: {amb_final.shape}')

Ambulance — independent: 64, fallback: 9
Ambulance final shape: (1701, 5)


## 4. NHS 111

111 data is published at 111 area level (53 areas), not CCG level. The same population-share stratification approach is used as for ambulance. The mapping from CCG to 111 area comes from a sheet inside the MDS time series file.

48 of 73 CCGs can be independently processed. The remaining 25 are not present in the 111 mapping sheet. Likely reflecting CCGs that were served by providers not captured in the mapping, or areas reorganised after the mapping was created.

In [16]:
def extract_111(filepath):
    df = pd.read_excel(filepath, sheet_name='Raw', header=None)
    data = df.iloc[6:].copy().reset_index(drop=True)
    data = data[[3,4,5,7,9]].copy()
    data.columns = ['period','area_code','area_name','111_offered_raw','111_answered_raw']
    data = data.dropna(subset=['period'])
    data['period'] = pd.to_datetime(data['period'])
    data['month'] = data['period'].dt.strftime('%b')
    data['year']  = data['period'].dt.year
    data = data[data['year'].isin([2018,2019])]
    data['111_offered_raw']  = pd.to_numeric(data['111_offered_raw'],  errors='coerce')
    data['111_answered_raw'] = pd.to_numeric(data['111_answered_raw'], errors='coerce')
    return data

def get_111_ccg_mapping(filepath):
    df = pd.read_excel(filepath, sheet_name='CCG to 111 Area & Provider', header=None)
    mapping = df.iloc[2:,[1,4]].copy()
    mapping.columns = ['ccg','area_code']
    mapping = mapping.dropna()
    mapping = mapping[mapping['ccg'].str.match(r'^[0-9A-Z]{3}$', na=False)]
    return mapping

In [17]:
data_111    = extract_111(NHS111_FILE)
mapping_111 = get_111_ccg_mapping(NHS111_FILE)
print(f'111 raw: {data_111.shape}, areas: {data_111["area_code"].nunique()}')

111 raw: (915, 7), areas: 53


In [18]:
data_111_ccg = data_111.merge(mapping_111, on='area_code', how='left')
data_111_ccg = data_111_ccg[data_111_ccg['ccg'].isin(master_ccg_list)].copy()
data_111_ccg = data_111_ccg.merge(pop_complete[['ccg','year','population']], on=['ccg','year'], how='left')

area_pop = data_111_ccg.groupby(['area_code','month','year'])['population'].transform('sum')
data_111_ccg['pop_share'] = data_111_ccg['population'] / area_pop
data_111_ccg['111_111_offered']  = data_111_ccg['111_offered_raw']  * data_111_ccg['pop_share'] / data_111_ccg['population']
data_111_ccg['111_111_answered'] = data_111_ccg['111_answered_raw'] * data_111_ccg['pop_share'] / data_111_ccg['population']

In [19]:
nhs111_independent = data_111_ccg[['ccg','month','year','111_111_offered','111_111_answered']].dropna()
independent_111_ccgs = set(nhs111_independent['ccg'].unique())
fallback_111_ccgs = set(master_ccg_list) - independent_111_ccgs
print(f'111 — independent: {len(independent_111_ccgs)}, fallback: {len(fallback_111_ccgs)}')

nhs111_fallback = master[master['ccg'].isin(fallback_111_ccgs)][['ccg','month','year','111_111_offered','111_111_answered']].copy()
nhs111_final = pd.concat([nhs111_independent, nhs111_fallback], ignore_index=True)
print(f'111 final shape: {nhs111_final.shape}')

111 — independent: 48, fallback: 25
111 final shape: (1499, 5)


## 5. GP Appointments

GP appointment data is published by NHS Digital at CCG level monthly, so no stratification is needed. The total count of appointments is extracted from Table 4 of each monthly publication file and scaled by population.

This publication only started in October 2018, so January-September 2018 values are not available from this source. Those months are filled from the authors' master file in the merge step.

In [20]:
def extract_gp(filepath, month, year):
    df = pd.read_excel(filepath, sheet_name='Table 4', header=None)
    df.columns = df.iloc[10]
    df = df.iloc[11:].reset_index(drop=True)
    ccg = df[df['Type'] == 'CCG'].copy()
    result = pd.DataFrame()
    result['ccg'] = ccg['NHS Area Code'].values
    result['gp_appt_available'] = pd.to_numeric(ccg['Total Count of Appointments'], errors='coerce').values
    result['month'] = month
    result['year']  = year
    return result

def parse_gp_filename(filepath):
    name = os.path.basename(filepath).replace('.xlsx','').replace('.xls','')
    months = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']
    month, year = None, None
    for m in months:
        if m.lower() in name.lower():
            month = m[:3]; break
    match = re.search(r'(2018|2019)', name)
    if match: year = int(match.group(1))
    return month, year

In [21]:
all_gp = []
for f in sorted(glob.glob(os.path.join(GP_PATH,'*.xlsx'))):
    month, year = parse_gp_filename(f)
    if month is None or year is None: print(f'Skipping {os.path.basename(f)}'); continue
    try:
        gp = extract_gp(f, month, year)
        all_gp.append(gp)
        print(f'Processed {month} {year}: {len(gp)} CCGs')
    except Exception as e:
        print(f'Error in {os.path.basename(f)}: {e}')

Processed Apr 2019: 190 CCGs
Processed Aug 2019: 191 CCGs
Processed Dec 2019: 191 CCGs
Skipping GP_APPT_Publication_February _2022.xlsx
Skipping GP_APPT_Publication_January _2022.xlsx
Processed Jul 2019: 191 CCGs
Processed Jun 2019: 191 CCGs
Skipping GP_APPT_Publication_March_2022_V2.xlsx
Processed May 2019: 190 CCGs
Processed Nov 2019: 191 CCGs
Processed Oct 2019: 191 CCGs
Processed Sep 2019: 191 CCGs
Skipping GP_Appointment_Publication_Summary_April_2024.xlsx
Skipping GP_Appointment_Publication_Summary_August_2023.xlsx
Skipping GP_Appointment_Publication_Summary_August_2024.xlsx
Skipping GP_Appointment_Publication_Summary_December_2023.xlsx
Skipping GP_Appointment_Publication_Summary_December_2024.xlsx
Skipping GP_Appointment_Publication_Summary_February_2024.xlsx
Skipping GP_Appointment_Publication_Summary_January_2024.xlsx
Skipping GP_Appointment_Publication_Summary_July_2023.xlsx
Skipping GP_Appointment_Publication_Summary_July_2024.xlsx
Skipping GP_Appointment_Publication_Summary

In [22]:
gp_df = pd.concat(all_gp, ignore_index=True)
gp_filtered = gp_df[gp_df['ccg'].isin(master_ccg_list)].copy()
gp_filtered = gp_filtered.merge(pop_complete[['ccg','year','population']], on=['ccg','year'], how='left')
gp_filtered['gp_appt_available'] = gp_filtered['gp_appt_available'] / gp_filtered['population']
gp_final = gp_filtered[['ccg','month','year','gp_appt_available']].dropna()
print(f'GP final shape: {gp_final.shape} | CCGs: {gp_final["ccg"].nunique()}')

GP final shape: (708, 4) | CCGs: 48


## 6. Merge

All datasets are merged on CCG code, month and year. Using ED attendances as the base, every CCG-month in the authors' dataset gets a row. GP values missing for January-September 2018 (before the publication started) are filled from the authors' master file. Rows with any remaining missing values are dropped.

In [23]:
df = ed.copy()
df = df.merge(pop_complete, on=['ccg','year'], how='left')
df = df.merge(gp_final,    on=['ccg','month','year'], how='left')
df = df.merge(amb_final,   on=['ccg','month','year'], how='left')
df = df.merge(nhs111_final,on=['ccg','month','year'], how='left')
df = df.merge(health_index,on=['ccg','year'], how='left')

print('Before GP fill — missing gp_appt_available:', df['gp_appt_available'].isna().sum())

Before GP fill — missing gp_appt_available: 813


In [24]:
# fill Jan-Sep 2018 GP values from authors' master
df = df.merge(gp_master, on=['ccg','month','year'], how='left', suffixes=('','_master'))
df['gp_appt_available'] = df['gp_appt_available'].fillna(df['gp_appt_available_master'])
df = df.drop(columns=['gp_appt_available_master'])

print('After GP fill — missing gp_appt_available:', df['gp_appt_available'].isna().sum())

df_clean = df.dropna()
print('Final shape:', df_clean.shape)
print('CCGs:', df_clean['ccg'].nunique())

After GP fill — missing gp_appt_available: 0
Final shape: (1435, 13)
CCGs: 73


## 7. Verify

A verification comparing our processed values against the authors' for CCG 00Q. GP and ambulance should match exactly for the independently processed CCGs.

In [25]:
test_ccg = '00Q'
yours  = df_clean[df_clean['ccg']==test_ccg].sort_values(['year','month'])
theirs = master[master['ccg']==test_ccg].sort_values(['year','month'])

print(f'Rows — ours: {len(yours)}, authors: {len(theirs)}')
print()
print('GP:')
print('  Ours:   ', yours['gp_appt_available'].values[:6])
print('  Authors:', theirs['gp_appt_available'].values[:6])
print()
print('Ambulance:')
print('  Ours:   ', yours['amb_sys_made'].values[:6])
print('  Authors:', theirs['amb_sys_made'].values[:6])

Rows — ours: 24, authors: 24

GP:
  Ours:    [3974.43300077 4025.72813578 3836.9298116  3910.91834405 4568.01976608
 4414.73862309]
  Authors: [3974.43300077 4025.72813578 3836.9298116  3910.91834405 4568.01976608
 4414.73862309]

Ambulance:
  Ours:    [264.97318059 274.16507322 285.20284457 261.75643485 310.56180131
 297.86982263]
  Authors: [264.97318059 274.16507322 285.20284457 261.75643485 310.56180131
 297.86982263]


## 8. Save

In [26]:
output_path = os.path.join(OUTPUT_PATH, 'master_scaled_replicated.csv')
df_clean.to_csv(output_path, index=True)
print(f'Saved: {output_path}')
print(f'Shape: {df_clean.shape}')

Saved: ../new_data/master_scaled_replicated.csv
Shape: (1435, 13)
